In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In this notebook I used time-resolved mixed-effects regression to identify potential "biomarkers" - patterns of HFA that:

1. Change consistently wrt behavior across electrodes, across subjects, while accounting for inter-electrode differences in intercept
2. Survive permutation correction (trial-shifting)
3. Survive multiple-comparison correction across time (FDR-BY)

I then run single-electrode, permutation-corrected regressions and characterize the "biomarker" rank of each electrode, both within subjects and across all electrodes. 

In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
from statsmodels.stats.multitest import multipletests
from mne.stats import permutation_cluster_1samp_test
from scipy.signal import filtfilt
from joblib import Parallel, delayed
import os

# Import plotting functions
# Import the FOOOF object
from tqdm import tqdm
import warnings

# I only want to see warnings once
warnings.filterwarnings('ignore')

plot_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-TimeResolvedRegression_Template'
os.makedirs(plot_dir, exist_ok=True)

from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df
from membandit._impl.multivariate import process_subject_smoothed_regression


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')


In [ ]:
from LFPAnalysis import statistics_utils


In [ ]:
from membandit.plotting_style import set_plot_style
set_plot_style()

## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:

learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

combined_df = combined_df[combined_df.condition=='Day1']


3/18/2025: Make all the time-wise regression plots:

In [ ]:
# # Load an example subject using the subject class from mv_regression_utils.py
# from membandit.io import Subject
# load_dir = '/Volumes/T7 Shield/work/qasims01/MemoryBanditData/EMU'
# subject = 'MS036'
# subject_obj = Subject(subject, base_dir, load_dir)

# subject_rows = []

# rois= ['HPC']
# event = 'feedback_start'
# predictor = 'zrpe'
# covariates = ['reward']
# regressor_source = None
# trial_var_override = None
# freq_range = (70., 201.)
# downsample_factor = 5
# win_ms = 200

# formula = "tfr ~ 1 + zrpe + reward"
# n_permutations = 100
# seed = 42

# import membandit._impl.multivariate

# for roi in rois:
#     try:
#         signal, times_sec, labels = mv_regression_utils._get_smoothed_event_signal(
#             subject_obj=subject_obj,
#             roi=roi,
#             event=event,
#             freq_range=freq_range,
#             downsample_factor=downsample_factor,
#             win_ms=win_ms
#         )
#         regressors, _ = mv_regression_utils._build_subject_regressors(
#             subject_obj=subject_obj,
#             event=event,
#             predictor=predictor,
#             covariates=covariates,
#             regressor_source=regressor_source,
#             trial_var_override=trial_var_override,
#         )
#         cache_key = (event, roi, tuple(freq_range), int(downsample_factor), int(win_ms))
#         neural_meta = subject_obj._neural_cache.get(cache_key, {})
#         hemi_raw = neural_meta.get('electrode_hemi', None)
#         region_raw = neural_meta.get('electrode_region', None)
#         hemi_arr = np.asarray(hemi_raw, dtype=object) if hemi_raw is not None else np.array([], dtype=object)
#         region_arr = np.asarray(region_raw, dtype=object) if region_raw is not None else np.array([], dtype=object)
#         if len(hemi_arr) == 0 or len(region_arr) == 0:
#             elec_sub = subject_obj.elec_df[
#                 subject_obj.elec_df.salman_region == roi
#             ][['label', 'salman_region', 'hemisphere']].copy()
#             elec_sub['label'] = elec_sub['label'].astype(str)
#             elec_sub = elec_sub.set_index('label')
#             region_fill = []
#             hemi_fill = []
#             for ch in labels:
#                 ch_key = str(ch)
#                 if ch_key in elec_sub.index:
#                     row = elec_sub.loc[ch_key]
#                     if isinstance(row, pd.DataFrame):
#                         row = row.iloc[0]
#                     region_fill.append(str(row.get('salman_region', roi)))
#                     hemi_val = row.get('hemisphere', np.nan)
#                     hemi_fill.append(str(hemi_val).lower() if pd.notna(hemi_val) else np.nan)
#                 else:
#                     region_fill.append(roi)
#                     hemi_fill.append(np.nan)
#             region_arr = np.asarray(region_fill, dtype=object)
#             hemi_arr = np.asarray(hemi_fill, dtype=object)
#     except Exception:
#         continue

#     for e_idx, label in enumerate(labels):
#         for t_idx, ts_sec in enumerate(times_sec):
#             ch_df = regressors.copy()
#             trial_ix = ch_df['trial'].values - 1
#             valid = (trial_ix >= 0) & (trial_ix < signal.shape[0])
#             if not np.any(valid):
#                 continue
#             ch_df = ch_df.loc[valid].copy()
#             trial_ix = trial_ix[valid]
#             ch_df['tfr'] = signal[trial_ix, e_idx, t_idx]
#             results =  mv_regression_utils.permutation_regression_zscore_local(
#                 ch_df, formula=formula, n_permutations=n_permutations, seed=seed
#             )
#             if len(results) == 0:
#                 continue
#             results['ts'] = float(ts_sec * 1000.0)
#             results['subj'] = subject
#             results['unique_label'] = label
#             results['region'] = (
#                 region_arr[e_idx] if e_idx < len(region_arr) else roi
#             )
#             results['hemi'] = (
#                 hemi_arr[e_idx] if e_idx < len(hemi_arr) else np.nan
#             )
#             subject_rows.append(results)



In [ ]:
# Smoothed time-resolved regression (complementary to binning approach above)
# Uses filtfilt smoothing with 200ms window and downsamples by factor of 5
# OPTIMIZED: Parallelized across subjects with checkpointing

import re

freqs = np.logspace(*np.log10([2, 200]), num=30)

freq_dict = {'theta':[4, 8],
            'alpha':[8, 13],
            'beta': [13, 30],
            'hfa': [70, 201]}

rois = ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']
day = 'Day1'
y='tfr'
# Smoothing parameters (HFA)
win_ms = 200  # 200ms smoothing window
downsample_factor = 5  # Downsample by factor of 5 after smoothing
nperm = 500

# Smoothing parameters (theta)


# Covariate config
COVARIATES = ['preward', 'zpm']
COVARIATE_FILENAME_MODE = 'append_covars'

# Option to overwrite existing checkpoints (set to True to re-run all subjects)
overwrite_checkpoints = True


def build_covariate_tag(covariates):
    covars_sorted = sorted(set(covariates or []))
    if len(covars_sorted) == 0:
        return 'none', covars_sorted
    safe_tokens = []
    for cov in covars_sorted:
        token = re.sub(r'[^A-Za-z0-9_-]+', '_', str(cov)).strip('_')
        safe_tokens.append(token if token else 'cov')
    return '-'.join(safe_tokens), covars_sorted


# Main processing loop with parallelization and checkpointing
for event in ['cue_on']:
    for predictor in ['zrpe']:
        covars_tag, covars_sorted = build_covariate_tag(COVARIATES)
        if predictor == 'rpe_cat':
            formula = "tfr ~ 1 + C(rpe_cat, Treatment(reference='neutral'))"
            if len(covars_sorted) > 0:
                formula = formula + " + " + " + ".join(covars_sorted)
        else:
            rhs_terms = [predictor] + covars_sorted
            formula = f"tfr ~ 1 + {' + '.join(rhs_terms)}"

        for band in ['hfa']:
            # Create checkpoint directory for intermediate saves
            if COVARIATE_FILENAME_MODE == 'append_covars':
                checkpoint_dir = f'{load_dir}/checkpoints_smoothed_{band}_{event}_{predictor}_cov-{covars_tag}'
            else:
                checkpoint_dir = f'{load_dir}/checkpoints_smoothed_{band}_{event}_{predictor}'
            os.makedirs(checkpoint_dir, exist_ok=True)

            # Check for existing checkpoints (unless overwrite is enabled)
            completed_subjects = []
            if not overwrite_checkpoints:
                for subj in subj_ids:
                    checkpoint_file = f'{checkpoint_dir}/{subj}_results.csv'
                    if os.path.exists(checkpoint_file):
                        completed_subjects.append(subj)
                        print(f"Found checkpoint for {subj}, skipping...")
            else:
                print("Overwrite mode enabled: will re-process all subjects")

            # Get subjects that still need processing
            remaining_subjects = [s for s in subj_ids if s not in completed_subjects]

            if len(remaining_subjects) == 0:
                print("All subjects already processed. Loading from checkpoints...")
            else:
                print(f"Processing {len(remaining_subjects)} subjects (skipping {len(completed_subjects)} already completed)...")

                # Parallel processing with limited jobs to avoid memory issues
                n_jobs = min(8, os.cpu_count())
                print(f"Using {n_jobs} parallel workers...")

                all_results = Parallel(n_jobs=n_jobs, verbose=1)(
                    delayed(process_subject_smoothed_regression)(
                        subject=subj,
                        event=event,
                        predictor=predictor,
                        formula=formula,
                        base_dir=base_dir,
                        load_dir=load_dir,
                        rois=rois,
                        freq_range=tuple(freq_dict[band]),
                        win_ms=win_ms,
                        downsample_factor=downsample_factor,
                        n_permutations=nperm,
                        covariates=covars_sorted,
                        min_trials=10,
                        seed=42,
                        regressor_source=None,
                        trial_var_override=None,
                    )
                    for subj in remaining_subjects
                )

                # Save checkpoints for each subject
                print("Saving checkpoints...")
                for subj, subject_df in zip(remaining_subjects, all_results):
                    if isinstance(subject_df, pd.DataFrame) and len(subject_df) > 0:
                        checkpoint_file = f'{checkpoint_dir}/{subj}_results.csv'
                        subject_df.to_csv(checkpoint_file, index=False)
                        print(f"  Saved checkpoint for {subj}: {len(subject_df)} rows")

            # Load all results from checkpoints
            print("Loading all results from checkpoints...")
            all_model_dfs = []
            for subj in subj_ids:
                checkpoint_file = f'{checkpoint_dir}/{subj}_results.csv'
                if os.path.exists(checkpoint_file):
                    subj_df = pd.read_csv(checkpoint_file)
                    all_model_dfs.append(subj_df)
                    print(f"  Loaded {subj}: {len(subj_df)} rows")

            # Concatenate all results
            if len(all_model_dfs) > 0:
                model_dfs = pd.concat(all_model_dfs, ignore_index=True)
                model_dfs['covariates'] = ','.join(covars_sorted)
                model_dfs['formula'] = formula
                print(f"Total results: {len(model_dfs)}")

                # Save final combined results
                if COVARIATE_FILENAME_MODE == 'append_covars':
                    output_file = f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_cov-{covars_tag}_smoothed_final.csv'
                else:
                    output_file = f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_smoothed_final.csv'
                model_dfs.to_csv(output_file, index=False)
                print(f"Saved final results to: {output_file}")
            else:
                print("No results to save!")


In [ ]:
# subj = 'UI006'
# test_res = process_subject_smoothed_regression(
#                         subject=subj,
#                         event=event,
#                         predictor=predictor,
#                         formula=formula,
#                         base_dir=base_dir,
#                         load_dir=load_dir,
#                         rois=rois,
#                         freq_range=tuple(freq_dict[band]),
#                         win_ms=win_ms,
#                         downsample_factor=downsample_factor,
#                         n_permutations=nperm,
#                         covariates=covars_sorted,
#                         min_trials=10,
#                         seed=42,
#                         regressor_source=None,
#                         trial_var_override=None,
#                     )

In [ ]:
# Plotting code for smoothed time-resolved regression (complementary to binned analysis plotting above)

from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test

if 'build_covariate_tag' not in globals():
    import re
    def build_covariate_tag(covariates):
        covars_sorted = sorted(set(covariates or []))
        if len(covars_sorted) == 0:
            return 'none', covars_sorted
        safe_tokens = []
        for cov in covars_sorted:
            token = re.sub(r'[^A-Za-z0-9_-]+', '_', str(cov)).strip('_')
            safe_tokens.append(token if token else 'cov')
        return '-'.join(safe_tokens), covars_sorted

covars_tag, covars_sorted = build_covariate_tag(COVARIATES if 'COVARIATES' in globals() else ['zpm', 'pgamble'])

# Note: For smoothed data, timepoints are already in ms and downsampled
# No need for win_len/sr conversion since ts values are already in ms

hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059, 0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])

for event in ['cue_on']:
    for predictor in ['zrpe']:
        for band in ['hfa']:
            # Load smoothed analysis results (covariate-aware filename with legacy fallback)
            tagged_file = f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_cov-{covars_tag}_smoothed_final.csv'
        #     legacy_file = f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_smoothed_final.csv'
            load_file = tagged_file 
            # if os.path.exists(tagged_file) else legacy_file
            model_dfs = pd.read_csv(load_file)
            print(f'Loaded: {load_file}')
            fig, ax = plt.subplots(2, 3, figsize=(20, 10))
            for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
                # plot the single-electrode encoding data 
                
                if region=='OFC':
                        color = ofc_c
                        ix = 0
                        jx = 0
                elif region=='HPC': 
                        color = hpc_c
                        ix = 0
                        jx = 2
                elif region=='AMY':
                        color = amy_c
                        ix = 1
                        jx = 2                       
                elif region=='dmPFC':
                        color = dmpfc_c
                        ix = 1
                        jx = 0
                elif region=='dlPFC':
                        color = dlpfc_c
                        ix = 1
                        jx = 1
                elif region=='ACC':
                        color = acc_c
                        ix = 0
                        jx = 1
                

                ts = model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]['ts']
                
                # Get sorted unique timepoints - use this consistently for everything
                unique_ts = np.sort(model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]['ts'].unique())
                
                # Calculate mean and standard error for each timepoint
                region_data = model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]
                mean_beta = region_data.groupby('ts')['z_beta'].mean().loc[unique_ts].values
                se_beta = region_data.groupby('ts')['z_beta'].sem().loc[unique_ts].values
                
                # Plot line with error shading
                ax[ix, jx].plot(unique_ts, mean_beta, color=color, linewidth=1.5)
                ax[ix, jx].fill_between(unique_ts, mean_beta - se_beta, mean_beta + se_beta, 
                                       color=color, alpha=0.2)
                
                # For smoothed data, timepoints are already in ms, so adjust tick labels accordingly
                if event in ['cue_on', 'choice_time']:
                       # Use actual timepoints from data
                       xticks = unique_ts[::max(1, len(unique_ts)//5)]  # Sample ~5 ticks
                       xticklabels = [f'{int(t)}' for t in xticks]
                       xlim = [min(unique_ts) - 100, max(unique_ts) + 100]
                elif event == 'feedback_start':
                       # Use actual timepoints from data
                       xticks = unique_ts[::max(1, len(unique_ts)//7)]  # Sample ~7 ticks
                       xticklabels = [f'{int(t)}' for t in xticks]
                       xlim = [min(unique_ts) - 100, max(unique_ts) + 100]

                ax[ix, jx].set(xticks=xticks, 
                        xticklabels=xticklabels, 
                        xlim = xlim,
                        xlabel='time (ms)', ylabel=f'{predictor} (z)')
                sns.despine()
                
                # Run permutation test - reshape data using sorted unique timepoints
                T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
                        region_data.z_beta.values.reshape(-1, len(unique_ts)),
                        n_permutations=5000,
                        tail=0)
                
                if len(clusters)>0:
                        for i_c, c in enumerate(clusters):
                                if cluster_p_values[i_c] <= 0.05:
                                        # Convert cluster indices to actual time values
                                        # Clusters are indices into the sorted unique_ts array
                                        c_start_idx = max(0, clusters[i_c][0][0] - 1)
                                        c_end_idx = min(len(unique_ts) - 1, clusters[i_c][0][-1] + 1)
                                        # Get the actual time values at these indices
                                        c_ts = unique_ts[c_start_idx:c_end_idx+1]
                                        if len(c_ts) > 0:
                                            ax[ix, jx].fill_between(c_ts, 0.525, 0.6, color=color, alpha=0.5)

                ax[ix, jx].set(ylim=[-0.6, 0.6], title=region)
                ax[ix, jx].hlines(0, xmin=min(unique_ts), xmax=max(unique_ts), color='black', linestyle='--', linewidth=1)
                if (ix==0):
                       ax[ix, jx].set(xlabel=None, xticklabels=[])
                if (jx>0):
                       ax[ix, jx].set(ylabel=None, yticklabels=[])
                if ((ix==0) & (jx==0)) | ((ix==1) & (jx==0)):
                       pass
                else:
                       ax[ix, jx].set(ylabel=None)


plt.tight_layout()
plt.savefig(f'{plot_dir}/time_resolved_single_{band}_{event}_{predictor}_smoothed_final.pdf', dpi=300)
                                

## Load data and do more follow up analyses 

In [ ]:
def _load_covariate_aware_result(band, event, predictor, covars_tag):
    tagged_file = f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_cov-{covars_tag}_smoothed_final.csv'
    # legacy_file = f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_smoothed_final.csv'
    load_file = tagged_file 
    print(f'Loaded: {load_file}')
    return pd.read_csv(load_file)

for band in ['hfa']:
    # Load smoothed analysis results
    covars = ['pgamble']
    covars_tag, covars_sorted = build_covariate_tag(covars)
    choice_EV_df = _load_covariate_aware_result(band, 'choice_time', 'zQ', covars_tag)
    covars  = ['pgamble', 'preward', 'zpm']
    covars_tag, covars_sorted = build_covariate_tag(covars)
    feedback_RPE_df = _load_covariate_aware_result(band, 'feedback_start', 'zQ', covars_tag)
    covars = ['preward', 'zpm']
    covars_tag, covars_sorted = build_covariate_tag(covars)
    cue_on_df = _load_covariate_aware_result(band, 'cue_on', 'zQ', covars_tag)

choice_EV_df = choice_EV_df[choice_EV_df.predictor=='zQ']
choice_EV_df['unique_label'] = choice_EV_df.apply(lambda x: x.subj + '_' + x.unique_label, axis=1)
feedback_RPE_df = feedback_RPE_df[feedback_RPE_df.predictor=='zQ']
feedback_RPE_df['unique_label'] = feedback_RPE_df.apply(lambda x: x.subj + '_' + x.unique_label, axis=1)
cue_on_df = cue_on_df[cue_on_df.predictor=='zQ']
cue_on_df['unique_label'] = cue_on_df.apply(lambda x: x.subj + '_' + x.unique_label, axis=1)


In [ ]:
merged_df_fb = feedback_RPE_df.groupby(['unique_label', 'region', 'subj']).mean(numeric_only=True).reset_index().merge(choice_EV_df.groupby(['unique_label', 'region', 'subj']).mean(numeric_only=True).reset_index(), 
on=['unique_label', 'region', 'subj'], suffixes=('_feedback', '_choice'))

merged_df_cue = cue_on_df.groupby(['unique_label', 'region', 'subj']).mean(numeric_only=True).reset_index().merge(choice_EV_df.groupby(['unique_label', 'region', 'subj']).mean(numeric_only=True).reset_index(), 
on=['unique_label', 'region', 'subj'], suffixes=('_cue', '_choice'))

merged_df_cue2 = cue_on_df.groupby(['unique_label', 'region', 'subj']).mean(numeric_only=True).reset_index().merge(feedback_RPE_df.groupby(['unique_label', 'region', 'subj']).mean(numeric_only=True).reset_index(), 
on=['unique_label', 'region', 'subj'], suffixes=('_cue', '_feedback'))

In [ ]:
import statsmodels.formula.api as smf

smf.mixedlm('z_beta_cue ~ z_beta_feedback', data=merged_df_cue2[merged_df_cue2.region=='ACC'], groups='subj').fit().summary()b
